# ORA

結果(AdjPval)->  
1    有機物質に対する反応(0.002)
2    化学物質に対する反応(0.008)
3    骨髄系白血球の遊走(0.025)  
となった  


# Import Libraries

In [ ]:
library("tidyverse")
options(repr.plot.width=5, repr.plot.height=5)#Default=7x7

BiocManager::install("org.Hs.eg.db", ask = FALSE)
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("org.Hs.eg.db")

BiocManager::install("clusterProfiler")

#Bioconductor
for (package in c("DEqMS","WGCNA")) {
    if (!requireNamespace("BiocManager", quietly=TRUE))
        # install.packages("BiocManager")
    BiocManager::install(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}
#CRAN
for (package in c("openxlsx", "readxl", "RColorBrewer", "ggrepel")) {
    # install.packages(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}



In [ ]:
library(ggplot2)
library(dplyr)

In [ ]:

library(org.Hs.eg.db)


# Import Datasets

In [ ]:
fileDir <- "./Data/"
fileName <- "pro_data_common_val_Assay.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data <- temp


fileDir <- "./Data/"
fileName <- "pro_sample_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path,row.names = 1)
head(temp)
pro_sample <- temp

fileDir <- "./Data/"
fileName <- "map_OID_Uni_Sym.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_analyte <- temp

fileDir <- "./Data/"
fileName <- "DEP_list.csv"

path <- file.path(fileDir, fileName)

temp <- read.csv(path,  check.names = FALSE)

DEP_combined <- temp

DEP_Early_vs_Control <- DEP_combined[
  DEP_combined$Contrast == "Early_vs_Control",
]

DEP_Advanced_vs_Control <- DEP_combined[
  DEP_combined$Contrast == "Advanced_vs_Control",
]

DEP_Advanced_vs_Early <- DEP_combined[
  DEP_combined$Contrast == "Advanced_vs_Early",
]
rownames(DEP_Early_vs_Control) <- DEP_Early_vs_Control$Protein
rownames(DEP_Advanced_vs_Control) <- DEP_Advanced_vs_Control$Protein
rownames(DEP_Advanced_vs_Early) <- DEP_Advanced_vs_Early$Protein

DEP_Early_vs_Control$Protein <- NULL
DEP_Advanced_vs_Control$Protein <- NULL
DEP_Advanced_vs_Early$Protein <- NULL

View(DEP_combined)
View(DEP_Early_vs_Control)
View(DEP_Advanced_vs_Control)
View(DEP_Advanced_vs_Early)


# ORA

# Enrichment

## BP(Biological Process)

In [ ]:
head(pro_data)

## Control vs Early

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_BP <- enrichGO(
    gene = rownames(DEP_Early_vs_Control),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "BP",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_BP)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

In [ ]:

temp <- as.data.frame(ego_BP)%>%
    dplyr::rename(ModuleID=ID, ModuleName=Description, Ratio2DEPs=GeneRatio, Ratio2BGs=BgRatio,
                  Pval=pvalue, AdjPval=p.adjust, Qval=qvalue, MappedDEP=geneID, nMappedDEPs=Count)

head(temp)
ego_df <- temp

Logの列を追加

In [ ]:
temp <- ego_df
temp$LogAdjPval <- -log10(temp$AdjPval)
head(temp)
ego_df <- temp

## Early vs Advanced

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_BP <- enrichGO(
    gene = rownames(DEP_Advanced_vs_Early),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "BP",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_BP)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

## Control vs Advanced

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_BP <- enrichGO(
    gene = rownames(DEP_Advanced_vs_Control),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "BP",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_BP)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

# MF

## Control vs Early

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_MF <- enrichGO(
    gene = rownames(DEP_Early_vs_Control),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "MF",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_MF)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

## Early vs Advanced

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_MF <- enrichGO(
    gene = rownames(DEP_Advanced_vs_Early),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "MF",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_MF)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

## Control vs Advanced

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_MF <- enrichGO(
    gene = rownames(DEP_Advanced_vs_Control),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "MF",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_MF)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

# CC

## Control vs Early

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_CC <- enrichGO(
    gene = rownames(DEP_Early_vs_Control),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "CC",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_CC)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

## Early vs Advanced

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_CC <- enrichGO(
    gene = rownames(DEP_Advanced_vs_Early),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "CC",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_CC)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

## Control vs Advanced

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_CC <- enrichGO(
    gene = rownames(DEP_Advanced_vs_Control),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "CC",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 100,
    keyType = "SYMBOL",
    readable = TRUE 
)
temp <- as.data.frame(ego_CC)

temp <- temp[, c("ID", "Description", "GeneRatio", "BgRatio")]
rownames(temp) <- rownames(as.data.frame(ego_BP))

head(temp)

## MF

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_MF <- enrichGO(
    gene = rownames(DEPlist),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "MF",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 10000,
    keyType = "SYMBOL",
    readable = TRUE 
)
head(as.data.frame(ego_MF))

## CC

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
ego_CC <- enrichGO(
    gene = rownames(DEPlist),
    universe = colnames(pro_data), 
    OrgDb = org.Hs.eg.db,
    ont = "CC",          
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 10000,
    keyType = "SYMBOL",
    readable = TRUE 
)
head(as.data.frame(ego_CC))

## EnrichGO　のプロット

In [ ]:
ego_all <- list(
    BP = ego_BP,
    MF = ego_MF,
    CC = ego_CC
)
library(ggplot2)

visualizeGoEnrichment <- function(enrichment_results) {

    combined_plot <- lapply(names(enrichment_results), function(ont_name) {

        ego <- enrichment_results[[ont_name]]

        df <- as.data.frame(ego)

        # 結果がない場合はスキップ
        if (nrow(df) == 0) {
            return(NULL)
        }

        # GeneRatioを数値に変換
        parts <- strsplit(as.character(df$GeneRatio), "/")

        df$GeneRatio <- sapply(
            parts,
            function(x) as.numeric(x[1]) / as.numeric(x[2])
        )

        # Ontology
        df$ont <- ont_name

        # p.adjustの小さい順に並べて上位10個
        df <- df[order(df$p.adjust), ]

        df <- head(df, 10)

        return(df)
    })

    # NULLを除外
    combined_plot <- Filter(Negate(is.null), combined_plot)

    # データを結合
    combined_plot <- do.call(rbind, combined_plot)

    # BP → MF → CC の順番
    combined_plot$ont <- factor(
        combined_plot$ont,
        levels = c("BP", "MF", "CC")
    )

    # GO termの順番
    combined_plot$Description <- reorder(
        combined_plot$Description,
        combined_plot$GeneRatio
    )

    # Plot
    ggplot(
        combined_plot,
        aes(
            x = GeneRatio,
            y = Description
        )
    ) +
        geom_point(
            aes(
                size = Count,
                color = -log10(p.adjust)
            )
        ) +
        scale_color_continuous(
            low = "blue",
            high = "red"
        ) +
        facet_grid(
            ont ~ .,
            scales = "free_y"
        ) +
        theme_bw() +
        theme(
            axis.text.x = element_text(
                angle = 45,
                hjust = 1
            )
        ) +
        labs(
            x = "Gene Ratio",
            y = "GO Term",
            color = "-log10(adjusted P-value)",
            size = "Count"
        )
}
p <- visualizeGoEnrichment(ego_all)

print(p)

# KEGG pathway

In [ ]:
head(DEPlist)
# head(pro_analyte)

In [ ]:
head(pro_analyte)
head(pro_data)

In [ ]:
temp$UniProt <- mapping_vec[rownames(DEPlist)]

data.frame(
    Symbol = rownames(DEPlist),
    UniProt = temp$UniProt
)

In [ ]:
temp <- DEPlist
mapping_vec <- setNames(pro_analyte$UniProt, pro_analyte$Symbol)
temp$UniProt <- mapping_vec[rownames(DEPlist)]
temp
universe_uniprot <- mapping_vec[colnames(pro_data)]
universe_uniprot <- universe_uniprot[!is.na(universe_uniprot)]
# 2. KEGG解析の実行
ekegg <- enrichKEGG(
    gene = temp$UniProt,
    universe = universe_uniprot, 
    organism = 'hsa',           # ヒト(Homo sapiens)を指定
    pAdjustMethod = "BH",
    pvalueCutoff = 1,
    qvalueCutoff = 1,
    minGSSize = 4,
    maxGSSize = 10000,
    keyType = "uniprot"
)

# 結果の確認
head(as.data.frame(ekegg))

In [ ]:
library(enrichplot)
options(repr.plot.width = 12, repr.plot.height = 12)

# Cnetplotの実行

ekegg2 <- setReadable(ekegg, 
                           OrgDb = org.Hs.eg.db, 
                           keyType = "UNIPROT") # もし元のIDがUniProtなら "UNIPROT"

# showCategoryで表示するパスウェイ数を指定（上位5つなど）
p <- cnetplot(
    ekegg2, 
    showCategory = 6, 
    circular = TRUE,    # 円状に配置して見やすくする
    colorEdge = TRUE)   # パスウェイごとに線を色分け
p

# Export

In [ ]:
#Save the workbook as one single .xlsx file
fileDir <- "./ExportedData/"

fileName_rds <- "clusterProfiler-results.rds"
fileName_csv <- "clusterProfiler-results.csv"

saveRDS(ego,file = file.path(fileDir, fileName_rds))


write.csv(ego_df,
          file = file.path(fileDir, fileName_csv),
          row.names = FALSE)


In [ ]:

##　わからない
    # temp <- ego %>%
    #     dplyr::filter(pvalue<0.05) %>%#Display only nominal P-value < 0.05
    #     dplyr::mutate(PvalLabel=str_c("AdjPval = ",scales::scientific(p.adjust, digits=2)),
    #                   AdjSignif=ifelse(p.adjust<0.05, "True", "False")) %>%
    #     barplot(., x="Count", color="p.adjust", showCategory=display) +
    #     geom_text(aes(label=PvalLabel, color=AdjSignif), nudge_x=0.5, hjust=0) +
    #     coord_cartesian(clip="off") +
    #     scale_x_continuous(limits=c(0, 11), breaks=seq(0, 10, by=2), expand=c(0, 0)) +
    #     scale_y_discrete(labels=function(x) {str_wrap(x, width=60)}) +
    #     scale_fill_viridis_c(begin=0, end=1, direction=1, option="plasma",
    #                          limits=c(0, 0.4), breaks=seq(0, 0.4, by=0.1), name="AdjPval") +
    #     scale_color_manual(values=c("True"="#990000", "False"="gray40"), ) +
    #     guides(fill=guide_colorbar(reverse=TRUE), color="none") +
    #     labs(x="Count of DEPs",
    #          y="", title="Barplot") +
    #     theme_classic(base_size=16, base_family="Helvetica") +
    #     theme(text=element_text(face="plain", color="black", family="Helvetica")) +
    #     theme(axis.text.x=element_text(face="plain", color="black", family="Helvetica"),
    #           axis.text.y=element_text(face="plain", color="black", family="Helvetica", lineheight=0.75),
    #           axis.title=element_text(face="plain", color="black", family="Helvetica")) +
    #     theme(plot.title=element_text(size=18, hjust=1.0)) +
    #     theme(legend.direction="vertical", legend.box="horizontal",
    #           legend.background=element_blank())
    # options(repr.plot.width=9, repr.plot.height=max(c(1+display*0.25, 2.5)))
    # plot(temp)


In [ ]:
options(repr.plot.width=10, repr.plot.height=12)

# 1. データフレームに変換し、上位30個を抽出
temp <- as.data.frame(ego) %>%
  # 補正p値の小さい順にソートして上位30個を選択
  arrange(p.adjust) %>%
  slice_head(n = 30) %>%
  # 可視化のために -log10(p.adjust) を計算
  mutate(log_p = -log10(p.adjust),
        nGeneRatio = sapply(GeneRatio, function(x) {
            nums <- as.numeric(strsplit(x, "/")[[1]])
            return(nums[1]/nums[2])
        })
         )

# 2. ggplot2で棒グラフを作成
p <- ggplot(temp, aes(x = reorder(Description, nGeneRatio), y = nGeneRatio, fill = p.adjust)) +
  geom_col() +
  coord_flip() +  # 横向きの棒グラフにする
  scale_fill_viridis_c() +  
  labs(
    x = "GO Terms",
    y = "-log10(Adjusted P-value)",
    fill = "-log10(Adj P-value)",
    title = "Enriched GO Terms (Top 30)"
  ) +
  theme_minimal()

print(p)

In [ ]:

p <- barplot(
    ego,
    showCategory = 30,
    orderBy = "AdjPval"
)
p 

In [ ]:
head(DEPlist)

In [ ]:
dim(ego)
head(ego)